# 03 — DistilBERT Fine-Tuning for SMS Spam Detection

## 1. Project Objective

The objective of this notebook is to fine-tune a pretrained DistilBERT transformer model to classify SMS messages as either legitimate (ham) or spam.

This experiment builds on the initial exploratory data analysis and the TF-IDF + Logistic Regression baseline developed earlier in the project.Untitled

## 2. Dataset

- **Source:** UCI SMS Spam Collection
- **Task:** Binary text classification
- **Input:** SMS message text
- **Target:** Message label (`ham` or `spam`)
- **Class mapping:** `ham = 0`, `spam = 1`
- **Dataset size after cleaning:** 5,169 messages

## 3. Experimental Workflow

1. Import required libraries and verify the environment.
2. Load the original dataset and reproduce the established cleaning procedure.
3. Split the dataset into training, validation, and test sets using stratified sampling.
4. Encode the target labels as numeric class IDs.
5. Load the pretrained DistilBERT tokenizer.
6. Tokenize the SMS messages and prepare input tensors.
7. Load the pretrained DistilBERT model with a sequence-classification head.
8. Fine-tune the model using the training set.
9. Monitor validation performance and select the best model checkpoint.
10. Evaluate the selected model on the held-out test set.
11. Compare the results with the TF-IDF + Logistic Regression baseline.
12. Save the trained model and tokenizer for later application integration.

## 4. Evaluation Metrics

The model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Particular attention will be given to spam recall and spam precision because both missed spam messages and legitimate messages incorrectly flagged as spam matter in a practical spam detector.

## 5. Reproducibility and Data Integrity

- Use a fixed random seed for reproducible data splitting and training where applicable.
- Use stratified sampling to preserve class proportions.
- Keep training, validation, and test data separate.
- Use the validation set for model-development decisions.
- Reserve the test set for final evaluation.
- Keep raw and processed datasets out of version control when excluded by `.gitignore`.
- Record model configuration, training settings, and evaluation results.

## 6. Expected Outcome

A fine-tuned DistilBERT spam classifier, a documented evaluation against the classical machine-learning baseline, and saved model artifacts that can later be integrated into a FastAPI application.


In [48]:
# Python standard library
import random
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)


In [49]:
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nAll packages imported successfully!")

PyTorch: 2.14.1+cpu
NumPy: 2.5.2
Pandas: 3.0.5
CUDA build: None
CUDA available: False

All packages imported successfully!


In [25]:
# Locate  raw dataset
data_path = Path("../data/raw/SMSSpamCollection")

# whether the file exists
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /home/student/email-spam-detector/data/raw/SMSSpamCollection


In [26]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

Dataset shape: (5572, 2)

First five rows:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [27]:
print("Dataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


In [28]:
# Convert text labels into numerical labels
label_mapping = {
    "ham": 0,
    "spam": 1
}

#creates a new column containing those numerical labels
df["label_id"] = df["label"].map(label_mapping)

# Inspect the conversion
print(df[["label", "label_id"]].head())

# Check for labels that were not mapped
print("\nMissing numerical labels:", df["label_id"].isnull().sum())

  label  label_id
0   ham         0
1   ham         0
2  spam         1
3   ham         0
4   ham         0

Missing numerical labels: 0


In [29]:
X = df["message"]
y = df["label_id"]

print("Messages:", X.shape)
print("Labels:", y.shape)

Messages: (5572,)
Labels: (5572,)


In [30]:
# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

Training samples: 3900
Validation samples: 836
Testing samples: 836
Total samples: 5572


In [31]:
# Check class proportions in each dataset split
print("Training distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training distribution:
label_id
0    0.865897
1    0.134103
Name: proportion, dtype: float64

Validation distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64

Testing distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64


In [52]:
# Load the pretrained DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully!
Vocabulary size: 30522


In [35]:
# Tokenize training messages
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize validation messages
val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize test messages
test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    max_length=128
)

In [36]:
print("Training messages:", len(train_encodings["input_ids"]))
print("Validation messages:", len(val_encodings["input_ids"]))
print("Test messages:", len(test_encodings["input_ids"]))

print("\nFirst training message token IDs:")
print(train_encodings["input_ids"][0])

print("\nFirst training message attention mask:")
print(train_encodings["attention_mask"][0])

Training messages: 3900
Validation messages: 836
Test messages: 836

First training message token IDs:
[101, 3125, 999, 9433, 1018, 1006, 2888, 1010, 1021, 1058, 6220, 1016, 2888, 7644, 2007, 1037, 3722, 2915, 2013, 1020, 4210, 2013, 1037, 3413, 2011, 15214, 27052, 2361, 2000, 2507, 9433, 1037, 1016, 3125, 7785, 2044, 6275, 8117, 2015, 1012, 102]

First training message attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [37]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        # Store the tokenized encodings in this object
        self.encodings = encodings

        # Convert labels to a Python list and store them
        self.labels = labels.tolist()

    def __len__(self):
        # Return the number of labels stored in this object
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve the input IDs for the message at index idx
        input_ids = self.encodings["input_ids"][idx]

        # Retrieve the attention mask for the message at index idx
        attention_mask = self.encodings["attention_mask"][idx]

        # Retrieve the label for the message at index idx
        label = self.labels[idx]

        # Convert input IDs into a PyTorch tensor
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        # Convert attention mask into a PyTorch tensor
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        # Convert the label into a PyTorch tensor
        label = torch.tensor(label, dtype=torch.long)

        # Return a dictionary containing the tensors
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }

In [41]:
train_dataset = SMSDataset(train_encodings, y_train)

val_dataset = SMSDataset(val_encodings, y_val)

test_dataset = SMSDataset(test_encodings, y_test)

print(
    "Training length:", len(train_dataset),
    "\nValidation length:", len(val_dataset),
    "\nTesting length:", len(test_dataset)
)

Training length: 3900 
Validation length: 836 
Testing length: 836


In [46]:
sample = train_dataset[0]

print(sample.items())
print(sample.values())
print(sample.keys())

dict_items([('input_ids', tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102])), ('attention_mask', tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])), ('labels', tensor(1))])
dict_values([tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [47]:
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label shape:", sample["labels"].shape)

Input IDs shape: torch.Size([41])
Attention mask shape: torch.Size([41])
Label shape: torch.Size([])


In [50]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [51]:
data_collator 

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [68]:
BATCH_SIZE = 16

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [71]:
batch = next(iter(train_dataloader))

print("\nIndividual values")
print(batch["input_ids"][:2])
print(batch["attention_mask"][:2])
print(batch["labels"][:2])
print("\nShapes")
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)


Individual values
tensor([[  101, 11333,  1012,  1012,  1012,  1057,  2061,  8114,  1012,  1012,
          1012, 20277,  1012,  1012,  1012,  2084,  2595,  1012,  1012,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0],
        [  101,  2655,  2033,  1010,  1045,  2572,  2741, 19466,  2013, 26236,
          9818,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0]]

In [ ]:
print("Real tokens in message 1:", batch["attention_mask"][0].sum().item())
print("Real tokens in message 2:", batch["attention_mask"][1].sum().item())